# Preparing MSG Cloud Mask Data

This notebook prepares cloud mask data from the EUMETSAT **MSG CLM** product (`EO:EUM:DAT:MSG:CLM`) for a single site. It:

1. Documents how the raw GRIB files were downloaded from the EUMETSAT Data Store.
2. Reads the local GRIB files and cuts out a small window around the site.
3. Stores the result as a NetCDF time series (`clm_site.nc`).

**Requirements**

- An EUMETSAT account with API credentials, provided as `EUMETSAT_KEY` and `EUMETSAT_SECRET` in a `.env` file.
- The `pygrib` package, which needs the ECCODES library.

## 0. Downloading the raw data (reference only)

Downloading roughly 5000 GRIB files takes a long time, so it ran overnight in a Docker container on a NAS using the [`eumdac`](https://pypi.org/project/eumdac/) command-line tool. This section is documentation only and is not executed by the notebook.

**1. Container (`docker-compose.yml`).** Credentials are read from the environment or an `.env` file and must never be committed.

```yaml
services:
  eumdac:
    image: python:3.12-slim
    container_name: eumdac
    environment:
      - EUMDAC_KEY=${EUMDAC_KEY}
      - EUMDAC_SECRET=${EUMDAC_SECRET}
      - PYTHONUNBUFFERED=1
    volumes:
      - /home/toIhr/eumdac:/data
      - /home/toIhr/eumdac/.eumdac:/root/.eumdac
    working_dir: /data
    restart: unless-stopped
    command: >
      sh -c "pip install eumdac &&
             eumdac set-credentials $$EUMDAC_KEY $$EUMDAC_SECRET &&
             sleep infinity"
```

**2. Download command.** Run inside the container; fetches only the `.grb` entries for the chosen period, using 3 parallel threads, and logs to `download.log`.

```bash
nohup eumdac download \
    --collection EO:EUM:DAT:MSG:CLM \
    --start 2025-05-12T00:00:00 --end 2025-06-30T23:59:59 \
    -o "/data/2025-05 to 06" \
    -t 3 --entry '*.grb' --yes \
    > /data/download.log 2>&1 &
```

## 1. Setup

In [ ]:
import os
from glob import glob

import eumdac as eumd
import numpy as np
import pygrib as pg
import xarray as xr
from dotenv import load_dotenv

# Authenticate against the EUMETSAT Data Store (credentials come from .env)
load_dotenv()
token = eumd.AccessToken((os.environ["EUMETSAT_KEY"], os.environ["EUMETSAT_SECRET"]))

## 2. Extract a site subset from local GRIB files

Collect all previously downloaded GRIB files (one per time step).

In [ ]:
files = glob("../data/cloud-prediction/2025-05 to 06/*/*.grb")

For each file, cut a 41 × 41 pixel window around the site (row `R`, column `C` in the full-disk grid, roughly 130 × 230 km) and keep the analysis time. Everything is then stacked into one `xarray.DataArray` sorted by time.

In [ ]:
# Site position in the full-disk grid (row, column)
R, C = 3354, 1637
# target = data[R, C]                  # the site itself
# box    = data[R-20:R+21, C-20:C+21]  # ~130 × 230 km surroundings

date_list = []
subset_list = []

for file in files:
    with pg.open(file) as fn:
        date = fn[1].analDate
        subset = fn[1].values[R-20:R+21, C-20:C+21]

    date_list.append(date)
    # Flip the x axis and store the cloud mask classes compactly as int8
    subset_list.append(subset[:, ::-1].copy().astype(np.int8))

clm_subset = xr.DataArray(
    np.stack(subset_list),
    dims=("time", "y", "x"),          # y = rows, x = columns
    coords={"time": np.array(date_list, dtype="datetime64[ns]")},
    name="clm",
)

clm_subset = clm_subset.sortby("time")

## 3. Save as NetCDF

Write the time series to disk for use in the later modelling notebooks.

In [ ]:
clm_subset.to_netcdf("../data/cloud-prediction/clm_site.nc")